# Phase 10 — ML Modeling

## Objective

The purpose of this phase is to develop machine learning approaches that can rank content items according to their relative opportunity for review or improvement.

The models will be compared against the simple baseline approaches established in Phase 9.

The main objectives are to:

- Prepare the modeling dataset.
- Define the model input features.
- Avoid target leakage and circular evaluation.
- Train suitable machine learning models.
- Generate opportunity scores for content items.
- Compare model rankings with the reference opportunity benchmark.
- Evaluate ranking performance using appropriate ranking metrics.

The model output should remain a relative opportunity score or ranking.

The model should not be interpreted as a causal predictor of future traffic, clicks, or Google rankings.

## 10.1 Modeling Strategy

The machine learning stage will focus on learning a ranking of content items according to their relative opportunity for review or improvement.

A key methodological concern is avoiding circular evaluation.

The reference opportunity score from Phase 9 was constructed using historical performance signals such as impressions, position, and CTR. Therefore, directly using the same aggregated performance signals as model inputs would make the model evaluation partially circular.

To reduce this risk, the modeling strategy will separate the information used to construct the opportunity benchmark from the information used as model inputs.

The modeling process will follow these principles:

1. Use the daily performance data to preserve the temporal structure of the dataset.
2. Define an earlier historical period as the feature period.
3. Define a later period as the evaluation period.
4. Construct the opportunity benchmark from the later period.
5. Use content characteristics and earlier available performance information as model inputs.
6. Train models using only information available before the evaluation period.
7. Generate predicted opportunity scores for the evaluation period.
8. Compare the resulting ranking against the later-period opportunity benchmark.

This temporal setup better reflects the intended real-world use case: identifying content that may deserve review based on information available before the evaluation period.

The models will be evaluated as ranking systems rather than as direct predictors of future Google rankings or traffic.

The main evaluation metrics will include:

- Precision@K
- Recall@K
- NDCG@K

Several top-K values will be considered, including:

- Top 10
- Top 25
- Top 50
- Top 100

Simple baselines from Phase 9 will be retained as reference points for comparison.


## 10.2 Target Construction

The modeling target should represent the opportunity of a content item during a future evaluation period.

Because the dataset does not contain a direct label indicating whether an optimization was successful, a future-period reference opportunity score will be used as the evaluation benchmark.

The target will be constructed from search-performance signals observed after the feature period.

The target construction will follow these principles:

- Features must come from an earlier period.
- The opportunity benchmark must come from a later period.
- No future-period performance information will be used as an input feature.
- The target will be defined at the `client_hash_id + content_hash_id` level.
- Content items with insufficient future exposure may be excluded from the benchmark.
- The target is an analytical opportunity benchmark, not a ground-truth label of successful optimization.

Before defining the final temporal split, the daily performance data will be inspected to determine suitable feature and evaluation periods.

In [3]:
import duckdb
import pandas as pd
import numpy as np

from huggingface_hub import list_repo_files

con = duckdb.connect()

con.execute("""
    PRAGMA temp_directory='C:/Users/anasm/AppData/Local/Temp/duckdb_tmp';
""")

con.execute("""
    PRAGMA memory_limit='1GB';
""")

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

files

['.gitattributes',
 'README.md',
 'dim_clients.parquet',
 'dim_content.parquet',
 'fact_content_daily_performance/month=2025-01/data_0.parquet',
 'fact_content_daily_performance/month=2025-02/data_0.parquet',
 'fact_content_daily_performance/month=2025-03/data_0.parquet',
 'fact_content_daily_performance/month=2025-04/data_0.parquet',
 'fact_content_daily_performance/month=2025-05/data_0.parquet',
 'fact_content_daily_performance/month=2025-06/data_0.parquet',
 'fact_content_daily_performance/month=2025-07/data_0.parquet',
 'fact_content_daily_performance/month=2025-08/data_0.parquet',
 'fact_content_daily_performance/month=2025-09/data_0.parquet',
 'fact_content_daily_performance/month=2025-10/data_0.parquet',
 'fact_content_daily_performance/month=2025-11/data_0.parquet',
 'fact_content_daily_performance/month=2025-12/data_0.parquet',
 'fact_content_daily_performance/month=2026-01/data_0.parquet',
 'fact_content_daily_performance/month=2026-02/data_0.parquet',
 'fact_content_daily_pe

In [4]:
from huggingface_hub import hf_hub_download

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

print(performance_file)

C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance_sample.parquet


In [5]:
con.execute(f"""
    CREATE OR REPLACE VIEW performance_deduplicated AS
    SELECT DISTINCT *
    FROM read_parquet('{performance_file}')
""")

print("Performance data loaded successfully.")

Performance data loaded successfully.


In [6]:
date_summary = con.execute("""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT report_date) AS number_of_days
    FROM performance_deduplicated
""").df()

date_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,number_of_days
0,2026-06-01,2026-06-30,30


In [7]:
daily_coverage = con.execute("""
    SELECT
        report_date,
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_content_items,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks
    FROM performance_deduplicated
    GROUP BY report_date
    ORDER BY report_date
""").df()

daily_coverage

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,row_count,unique_content_items,total_impressions,total_clicks
0,2026-06-01,390720,390720,8926939.0,46668.0
1,2026-06-02,391784,391784,8843550.0,52160.0
2,2026-06-03,392381,392381,8497881.0,48339.0
3,2026-06-04,392993,392993,8374318.0,45435.0
4,2026-06-05,393612,393612,7397007.0,40214.0
5,2026-06-06,394400,394400,7351130.0,37558.0
6,2026-06-07,395197,395197,7927389.0,44007.0
7,2026-06-08,395814,395814,7495281.0,45060.0
8,2026-06-09,396407,396407,7410511.0,43498.0
9,2026-06-10,397004,397004,7435021.0,37297.0


In [8]:
daily_coverage.describe(include='all')

,report_date,row_count,unique_content_items,total_impressions,total_clicks
count,30,30.000000,30.000000,3.000000e+01,30.000000
mean,2026-06-15 12:00:00,389589.400000,389589.400000,7.204526e+06,40294.233333
min,2026-06-01 00:00:00,339652.000000,339652.000000,5.958663e+06,24202.000000
25%,2026-06-08 06:00:00,392534.000000,392534.000000,6.748432e+06,35794.250000
50%,2026-06-15 12:00:00,397318.000000,397318.000000,7.054908e+06,40140.000000
75%,2026-06-22 18:00:00,401584.250000,401584.250000,7.428894e+06,45341.250000
max,2026-06-30 00:00:00,406239.000000,406239.000000,8.926939e+06,63135.000000
std,NaN,20876.387658,20876.387658,7.345797e+05,8043.354283


### 10.2.1 Temporal Split

Based on the available 30-day observation window, the data will be divided into two temporal periods:

**Feature Period**
- June 1, 2026 → June 20, 2026

**Evaluation Period**
- June 21, 2026 → June 30, 2026

The feature period will be used to construct information available before the evaluation period.

The evaluation period will be used to construct the future opportunity benchmark.

This temporal separation is intended to reduce look-ahead bias and better simulate the real-world use case of ranking content based on previously observed information.

In [9]:
feature_period_summary = con.execute("""
    SELECT
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_content_items,
        COUNT(DISTINCT report_date) AS number_of_days,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks
    FROM performance_deduplicated
    WHERE report_date BETWEEN DATE '2026-06-01'
                          AND DATE '2026-06-20'
""").df()

feature_period_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,unique_content_items,number_of_days,total_impressions,total_clicks
0,402701,20,147917622.0,792149.0


In [10]:
evaluation_period_summary = con.execute("""
    SELECT
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_content_items,
        COUNT(DISTINCT report_date) AS number_of_days,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks
    FROM performance_deduplicated
    WHERE report_date BETWEEN DATE '2026-06-21'
                          AND DATE '2026-06-30'
""").df()

evaluation_period_summary

,unique_content_items,number_of_days,total_impressions,total_clicks
0,409205,10,68218154.0,416678.0


In [11]:
period_overlap = con.execute("""
    SELECT
        COUNT(*) AS content_items_in_both_periods
    FROM (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM performance_deduplicated
        WHERE report_date BETWEEN DATE '2026-06-01'
                              AND DATE '2026-06-20'
    ) AS feature_items
    INNER JOIN (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM performance_deduplicated
        WHERE report_date BETWEEN DATE '2026-06-21'
                              AND DATE '2026-06-30'
    ) AS evaluation_items
    USING (client_hash_id, content_hash_id)
""").df()

period_overlap

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_items_in_both_periods
0,402701


### 10.2.2 Future Opportunity Target

The future opportunity benchmark will be constructed from the evaluation period only.

For each `client_hash_id + content_hash_id`, future search-performance signals will be aggregated over June 21–30, 2026.

The future benchmark will use:

- Total future impressions
- Total future clicks
- Mean future average position
- Future CTR

Only content items with at least 100 future impressions and a valid positive future average position will receive a reference opportunity score.

The same scoring logic established in Phase 9 will be applied to the future evaluation period.

This ensures that the benchmark represents opportunity observed after the feature period while preventing future information from entering the model inputs.

In [12]:
future_performance = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_impressions,
        SUM(gsc_clicks) AS future_clicks,
        AVG(gsc_avg_position) AS future_mean_position
    FROM performance_deduplicated
    WHERE report_date BETWEEN DATE '2026-06-21'
                          AND DATE '2026-06-30'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

future_performance['future_ctr'] = np.where(
    future_performance['future_impressions'] > 0,
    future_performance['future_clicks'] /
    future_performance['future_impressions'],
    np.nan
)

future_performance.shape

(409205, 6)

In [13]:
future_performance.head()

,client_hash_id,content_hash_id,future_impressions,future_clicks,future_mean_position,future_ctr
0,client_3ffa76342f366962,content_c1c1f7e879d6013b,0.0,0.0,NaN,NaN
1,client_3ffa76342f366962,content_f9becc7469a6d8a8,0.0,0.0,NaN,NaN
2,client_3ffa76342f366962,content_5f3ef4e8afff3fde,0.0,0.0,NaN,NaN
3,client_3ffa76342f366962,content_bf04bea1583cf0a5,0.0,0.0,NaN,NaN
4,client_3ffa76342f366962,content_177908c94195720a,0.0,0.0,NaN,NaN


In [14]:
future_reference = future_performance[
    (future_performance['future_impressions'] >= 100) &
    (future_performance['future_mean_position'].notna()) &
    (future_performance['future_mean_position'] > 0)
].copy()

print("Eligible future content items:", len(future_reference))

Eligible future content items: 65279


In [15]:
future_reference['log_impressions'] = np.log1p(
    future_reference['future_impressions']
)

future_reference['visibility_score'] = (
    future_reference['log_impressions'] /
    future_reference['log_impressions'].max()
)

future_reference['position_opportunity'] = np.select(
    [
        future_reference['future_mean_position'] <= 10,
        future_reference['future_mean_position'] <= 20,
        future_reference['future_mean_position'] <= 50,
        future_reference['future_mean_position'] > 50
    ],
    [0.25, 0.50, 0.75, 1.00],
    default=np.nan
)

future_reference['ctr_opportunity'] = (
    1 - future_reference['future_ctr']
)

future_reference['future_opportunity_score'] = (
    0.50 * future_reference['visibility_score']
    + 0.35 * future_reference['position_opportunity']
    + 0.15 * future_reference['ctr_opportunity']
)

future_reference = future_reference.sort_values(
    'future_opportunity_score',
    ascending=False
).reset_index(drop=True)

future_reference['future_reference_rank'] = (
    future_reference.index + 1
)

In [16]:
future_reference[
    [
        'client_hash_id',
        'content_hash_id',
        'future_impressions',
        'future_clicks',
        'future_mean_position',
        'future_ctr',
        'visibility_score',
        'position_opportunity',
        'ctr_opportunity',
        'future_opportunity_score',
        'future_reference_rank'
    ]
].head(20)

,client_hash_id,content_hash_id,future_impressions,future_clicks,future_mean_position,future_ctr,visibility_score,position_opportunity,ctr_opportunity,future_opportunity_score,future_reference_rank
0,client_3197e6291363b4db,content_65b8a4998e633d89,11490.0,0.0,81.077929,0.000000,0.754202,1.00,1.000000,0.877101,1
1,client_fef1a8f436438636,content_0aaa197051f58d6f,11246.0,3.0,55.296489,0.000267,0.752471,1.00,0.999733,0.876195,2
2,client_23a62021009f63c4,content_c60628276389acbb,7485.0,3.0,63.951936,0.000401,0.719633,1.00,0.999599,0.859756,3
3,client_3197e6291363b4db,content_3fd3671dd5e2604f,5094.0,0.0,87.062910,0.000000,0.688594,1.00,1.000000,0.844297,4
4,client_23a62021009f63c4,content_da36aaa1d72bdad4,4910.0,4.0,53.847444,0.000815,0.685626,1.00,0.999185,0.842691,5
5,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,4688.0,0.0,62.898324,0.000000,0.681895,1.00,1.000000,0.840947,6
6,client_e547b89c05043229,content_ba98b9ba325cb149,37823.0,19.0,41.066766,0.000502,0.850310,0.75,0.999498,0.837580,7
7,client_23a62021009f63c4,content_ea7474d92d9701c3,4092.0,8.0,54.880114,0.001955,0.670928,1.00,0.998045,0.835171,8
8,client_23a62021009f63c4,content_f75e5c8631965e3e,4094.0,12.0,54.672664,0.002931,0.670968,1.00,0.997069,0.835044,9
9,client_23a62021009f63c4,content_e22d7d712577870a,33662.0,1.0,21.374884,0.000030,0.840908,0.75,0.999970,0.832950,10


In [17]:
future_reference['future_opportunity_score'].describe()

count    65279.000000
mean         0.546534
std          0.076116
min          0.412835
25%          0.485568
50%          0.537198
75%          0.603533
max          0.877101
Name: future_opportunity_score, dtype: float64

## 10.3 Feature Construction

The model features will be constructed exclusively from the feature period:

June 1–20, 2026.

No performance information from the evaluation period will be used as a model input.

The feature dataset will be built at the:

`client_hash_id + content_hash_id`

level.

The features will include two main groups:

### Historical Performance Features

- Total impressions during the feature period
- Total clicks during the feature period
- Mean average position during the feature period
- Number of reporting days
- Historical CTR
- Impressions per reporting day
- Clicks per reporting day

### Content and Search Features

- Search volume
- Competition
- Competition level
- CPC
- Content type
- Main intent
- Backlinks
- Category count
- Word count
- Character count

Log-transformed versions of highly skewed numerical variables may also be included where appropriate.

The model will not use:

- Future-period performance
- Future opportunity score
- Future CTR
- Future position
- `client_hash_id` or `content_hash_id` as numerical predictive features
- Post-evaluation information

In [18]:
feature_performance = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS historical_impressions,
        SUM(gsc_clicks) AS historical_clicks,

        AVG(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                     AND gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS historical_mean_position,

        COUNT(DISTINCT report_date) AS historical_reporting_days

    FROM performance_deduplicated

    WHERE report_date BETWEEN DATE '2026-06-01'
                          AND DATE '2026-06-20'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feature_performance.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(402701, 6)

In [19]:
feature_performance['historical_ctr'] = np.where(
    feature_performance['historical_impressions'] > 0,
    feature_performance['historical_clicks'] /
    feature_performance['historical_impressions'],
    np.nan
)

feature_performance['historical_impressions_per_day'] = (
    feature_performance['historical_impressions'] /
    feature_performance['historical_reporting_days']
)

feature_performance['historical_clicks_per_day'] = (
    feature_performance['historical_clicks'] /
    feature_performance['historical_reporting_days']
)

feature_performance.head()

,client_hash_id,content_hash_id,historical_impressions,historical_clicks,historical_mean_position,historical_reporting_days,historical_ctr,historical_impressions_per_day,historical_clicks_per_day
0,client_b10cb2997d0c7c86,content_dd00cc9c59d75b7a,73.0,0.0,16.842708,20,0.0,3.65,0.0
1,client_b10cb2997d0c7c86,content_4bd2216df870737f,34.0,0.0,14.245000,20,0.0,1.70,0.0
2,client_b10cb2997d0c7c86,content_32d684c94f867c27,48.0,0.0,12.662222,20,0.0,2.40,0.0
3,client_b10cb2997d0c7c86,content_b71508b653e65838,0.0,0.0,NaN,20,NaN,0.00,0.0
4,client_b10cb2997d0c7c86,content_647a7152c3cf0edb,0.0,0.0,NaN,20,NaN,0.00,0.0


In [20]:
content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

content_file

'C:\\Users\\anasm\\.cache\\huggingface\\hub\\datasets--FlyRank--internship-warehouse\\snapshots\\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\\dim_content.parquet'

In [21]:
con.execute(f"""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT *
    FROM read_parquet('{content_file}')
""")

In [22]:
content_features = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        content_type,
        search_volume,
        competition,
        competition_level,
        cpc,
        main_intent,
        backlinks,
        category_count,
        char_count,
        word_count
    FROM dim_content
""").df()

content_features.shape

(519606, 12)

In [23]:
model_features = feature_performance.merge(
    content_features,
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

model_features.shape

(402701, 19)

In [24]:
modeling_dataset = model_features.merge(
    future_reference[
        [
            'client_hash_id',
            'content_hash_id',
            'future_opportunity_score',
            'future_reference_rank'
        ]
    ],
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

modeling_dataset.shape

(64765, 21)

In [25]:
# ----------------------
# ---Check duplicates---
# ----------------------

duplicate_count = modeling_dataset.duplicated(
    subset=['client_hash_id', 'content_hash_id']
).sum()

duplicate_count

0

In [26]:
modeling_dataset.isna().sum().sort_values(ascending=False)

backlinks                         16351
word_count                         7407
char_count                         7407
historical_mean_position            990
historical_ctr                      986
competition_level                   813
search_volume                       640
cpc                                 640
competition                         640
main_intent                         638
future_opportunity_score              0
category_count                        0
client_hash_id                        0
content_hash_id                       0
content_type                          0
historical_clicks_per_day             0
historical_impressions_per_day        0
historical_reporting_days             0
historical_clicks                     0
historical_impressions                0
future_reference_rank                 0
dtype: int64

In [27]:
modeling_dataset['future_opportunity_score'].describe()

count    64765.000000
mean         0.546482
std          0.075879
min          0.412835
25%          0.485778
50%          0.537269
75%          0.603266
max          0.877101
Name: future_opportunity_score, dtype: float64

In [28]:
# --------------------------
# ---Check target ranking---
# --------------------------

modeling_dataset[
    [
        'client_hash_id',
        'content_hash_id',
        'future_opportunity_score',
        'future_reference_rank'
    ]
].sort_values(
    'future_opportunity_score',
    ascending=False
).head(10)

,client_hash_id,content_hash_id,future_opportunity_score,future_reference_rank
2330,client_3197e6291363b4db,content_65b8a4998e633d89,0.877101,1
51745,client_fef1a8f436438636,content_0aaa197051f58d6f,0.876195,2
34828,client_23a62021009f63c4,content_c60628276389acbb,0.859756,3
35839,client_3197e6291363b4db,content_3fd3671dd5e2604f,0.844297,4
50979,client_23a62021009f63c4,content_da36aaa1d72bdad4,0.842691,5
41818,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,0.840947,6
63342,client_e547b89c05043229,content_ba98b9ba325cb149,0.837580,7
49886,client_23a62021009f63c4,content_ea7474d92d9701c3,0.835171,8
63015,client_23a62021009f63c4,content_f75e5c8631965e3e,0.835044,9
52574,client_23a62021009f63c4,content_e22d7d712577870a,0.832950,10


In [29]:
# ----------------------------
# Validate temporal separation
# ----------------------------

print("Feature period:")
print("2026-06-01 → 2026-06-20")

print("\nEvaluation period:")
print("2026-06-21 → 2026-06-30")

print("\nModeling dataset rows:", len(modeling_dataset))

Feature period:
2026-06-01 → 2026-06-20

Evaluation period:
2026-06-21 → 2026-06-30

Modeling dataset rows: 64765


## 10.3.1 Feature Construction Validation

The modeling dataset was constructed by combining historical features from the feature period with the future opportunity benchmark from the evaluation period.

The feature period covers June 1–20, 2026, while the evaluation period covers June 21–30, 2026.

The resulting dataset contains 64,765 content items at the `client_hash_id + content_hash_id` level.

The historical feature dataset contains 402,701 content items, while the future opportunity benchmark contains 65,279 eligible items. After the temporal merge, 64,765 items were retained for modeling.

The model inputs contain only information available during the feature period. The future opportunity score is used only as the evaluation target.

The dataset will therefore be passed to the feature preparation stage, where missing values, categorical variables, numerical variables, and feature transformations will be handled before model training.

## 10.4 Feature Preparation

The modeling dataset contains numerical and categorical features with different scales and missing-value patterns.

The preparation stage will transform these variables into a format suitable for machine learning while preserving the temporal separation established in the previous phase.

The preparation process will include:

- Selecting predictive features.
- Removing identifiers and target-related columns from the model inputs.
- Separating numerical and categorical features.
- Handling missing numerical values.
- Handling missing categorical values.
- Encoding categorical variables.
- Keeping the future opportunity score separate as the target.

The following identifiers will not be used as predictive features:

- `client_hash_id`
- `content_hash_id`

The following future-period columns will also be excluded from the model inputs:

- `future_opportunity_score`
- `future_reference_rank`

These columns are used only for evaluation.

No information from the evaluation period will be used during feature preparation or model training.

In [30]:
# ---------------------
# ---Define Features---
# ---------------------

target_column = 'future_opportunity_score'

numeric_features = [
    'historical_impressions',
    'historical_clicks',
    'historical_mean_position',
    'historical_reporting_days',
    'historical_ctr',
    'historical_impressions_per_day',
    'historical_clicks_per_day',
    'search_volume',
    'competition',
    'cpc',
    'backlinks',
    'category_count',
    'char_count',
    'word_count'
]

categorical_features = [
    'content_type',
    'competition_level',
    'main_intent'
]

feature_columns = numeric_features + categorical_features

X = modeling_dataset[feature_columns].copy()
y = modeling_dataset[target_column].copy()

X.shape, y.shape

((64765, 17), (64765,))

In [31]:
X[numeric_features].dtypes

historical_impressions            float64
historical_clicks                 float64
historical_mean_position          float64
historical_reporting_days           int64
historical_ctr                    float64
historical_impressions_per_day    float64
historical_clicks_per_day         float64
search_volume                       Int64
competition                       float64
cpc                               float64
backlinks                           Int64
category_count                      int64
char_count                          Int64
word_count                          Int64
dtype: object

In [33]:
X[categorical_features].dtypes

content_type         object
competition_level    object
main_intent          object
dtype: object

In [34]:
missing_before = X.isna().sum().sort_values(ascending=False)

missing_before[missing_before > 0]

backlinks                   16351
word_count                   7407
char_count                   7407
historical_mean_position      990
historical_ctr                986
competition_level             813
competition                   640
cpc                           640
search_volume                 640
main_intent                   638
dtype: int64

In [35]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical', categorical_transformer, categorical_features)
    ]
)

In [36]:
SimpleImputer(strategy='median')

SimpleImputer(strategy='median')

In [37]:
SimpleImputer(strategy='most_frequent')

SimpleImputer(strategy='most_frequent')

In [38]:
X_prepared = preprocessor.fit_transform(X)

X_prepared.shape

(64765, 26)

In [39]:
import numpy as np

print("Original shape:", X.shape)
print("Prepared shape:", X_prepared.shape)

print("Missing values after preparation:",
      np.isnan(X_prepared).sum())

print("Infinite values after preparation:",
      np.isinf(X_prepared).sum())

Original shape: (64765, 17)
Prepared shape: (64765, 26)
Missing values after preparation: 0
Infinite values after preparation: 0


### 10.4.1 Preparation Validation

The modeling features were successfully separated into numerical and categorical variables.

Numerical missing values were handled using median imputation, while categorical missing values were handled using the most frequent category.

Categorical variables were transformed using one-hot encoding with unknown categories ignored.

The client and content identifiers were excluded from the predictive feature set, and all future-period target information was kept separate from the model inputs.

The resulting prepared feature matrix contains no missing or infinite values and is ready for model training.

## 10.5 Model Training

A Random Forest Regressor will be used as the first machine learning model.

The model will learn the relationship between historical content and search-performance features from June 1–20, 2026 and the future opportunity benchmark observed during June 21–30, 2026.

Random Forest was selected as an initial model because it can capture non-linear relationships between features without requiring feature scaling.

The model will be trained to predict the continuous `future_opportunity_score`.

The predicted score will later be used to create a ranking of content items.

The model is not intended to predict future Google rankings, traffic, or the causal effect of optimization.

In [41]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

In [42]:
rf_model.fit(
    X_prepared,
    y
)

RandomForestRegressor(max_depth=12, min_samples_leaf=5, n_estimators=200,
                      n_jobs=-1, random_state=42)

In [43]:
train_predictions = rf_model.predict(X_prepared)

train_predictions[:10]

array([0.46742117, 0.50788505, 0.46563489, 0.62829899, 0.66245463,
       0.47937989, 0.47757011, 0.48589401, 0.62172197, 0.48711854])

In [44]:
pd.Series(train_predictions).describe()

count    64765.000000
mean         0.546473
std          0.064399
min          0.443372
25%          0.497836
50%          0.536444
75%          0.588594
max          0.834723
dtype: float64

In [45]:
np.corrcoef(
    train_predictions,
    y
)[0, 1]

0.8819204873670621

### 10.5.1 Training Result

The Random Forest model was successfully trained using the prepared historical feature matrix.

The model predicts a continuous future opportunity score for each content item.

Training predictions are used only as an implementation check. They are not considered the final evaluation because the model was trained on the same observations used to generate these predictions.

The final assessment will compare the model-generated ranking against the future opportunity benchmark using ranking-based evaluation metrics.

In [46]:
train_predictions[:10]

array([0.46742117, 0.50788505, 0.46563489, 0.62829899, 0.66245463,
       0.47937989, 0.47757011, 0.48589401, 0.62172197, 0.48711854])

In [47]:
pd.Series(train_predictions).describe()

count    64765.000000
mean         0.546473
std          0.064399
min          0.443372
25%          0.497836
50%          0.536444
75%          0.588594
max          0.834723
dtype: float64

In [48]:
np.corrcoef(
    train_predictions,
    y
)[0, 1]

0.8819204873670621

## 10.6 Ranking Prediction

The trained Random Forest model produces a continuous opportunity score for each content item.

These predicted scores will be converted into a ranking, where higher predicted scores indicate higher relative opportunity for review or improvement.

The ranking will be created at the `client_hash_id + content_hash_id` level.

The future opportunity score and future reference rank will be retained only for evaluation and will not be used to generate the model predictions.

In [49]:
ranking_predictions = modeling_dataset[
    [
        'client_hash_id',
        'content_hash_id',
        'future_opportunity_score',
        'future_reference_rank'
    ]
].copy()

ranking_predictions['predicted_opportunity_score'] = train_predictions

In [50]:
ranking_predictions['predicted_opportunity_score'] = rf_model.predict(
    X_prepared
)

In [51]:
ranking_predictions = ranking_predictions.sort_values(
    'predicted_opportunity_score',
    ascending=False
).reset_index(drop=True)

ranking_predictions['model_rank'] = (
    ranking_predictions.index + 1
)

In [52]:
ranking_predictions[
    [
        'client_hash_id',
        'content_hash_id',
        'predicted_opportunity_score',
        'future_opportunity_score',
        'model_rank',
        'future_reference_rank'
    ]
].head(20)

,client_hash_id,content_hash_id,predicted_opportunity_score,future_opportunity_score,model_rank,future_reference_rank
0,client_3197e6291363b4db,content_65b8a4998e633d89,0.834723,0.877101,1,1
1,client_23a62021009f63c4,content_c60628276389acbb,0.833381,0.859756,2,3
2,client_fef1a8f436438636,content_0aaa197051f58d6f,0.832910,0.876195,3,2
3,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,0.831638,0.840947,4,6
4,client_3197e6291363b4db,content_3fd3671dd5e2604f,0.821815,0.844297,5,4
5,client_23a62021009f63c4,content_ea7474d92d9701c3,0.816882,0.835171,6,8
6,client_e547b89c05043229,content_a788d093ce1ae235,0.812136,0.827906,7,11
7,client_23a62021009f63c4,content_ecbc72c4e65c4466,0.807634,0.755350,8,223
8,client_73cda7b4e4f265ea,content_3e56218fa52d24b8,0.807592,0.817884,9,12
9,client_23a62021009f63c4,content_2de9a39d3482a269,0.800527,0.813685,10,15


In [53]:
from scipy.stats import spearmanr

rank_correlation, rank_pvalue = spearmanr(
    ranking_predictions['model_rank'],
    ranking_predictions['future_reference_rank']
)

rank_correlation

0.8689247430738953

In [54]:
score_correlation, score_pvalue = spearmanr(
    ranking_predictions['predicted_opportunity_score'],
    ranking_predictions['future_opportunity_score']
)

score_correlation

0.8689243281855749

In [55]:
top_k_values = [10, 25, 50, 100]

top_k_rankings = {
    k: ranking_predictions.head(k).copy()
    for k in top_k_values
}

### 10.6.1 Ranking Output

The Random Forest model successfully generated a continuous opportunity score for each modeling item.

The scores were sorted in descending order to produce the final model ranking.

The ranking will be evaluated at multiple top-K levels, including Top 10, Top 25, Top 50, and Top 100.

The future opportunity score remains separate from the model prediction and is used only as the evaluation benchmark.

## 10.7 Model Evaluation

The model ranking will be evaluated against the future opportunity benchmark constructed from June 21–30, 2026.

Because the objective is to prioritize content items for review, the evaluation will focus on ranking quality rather than conventional regression accuracy.

The main evaluation metrics will be:

- Precision@K
- Recall@K
- NDCG@K

The evaluation will focus on:

- Top 10
- Top 25
- Top 50
- Top 100

A content item will be considered relevant when it belongs to the top-K items of the future opportunity benchmark.

The model ranking will be compared with temporal baselines constructed only from the historical feature period.

This prevents the evaluation baselines from using information from the future evaluation period.

In [56]:
import numpy as np

def precision_at_k(predicted_items, relevant_items, k):
    predicted_top_k = set(predicted_items[:k])
    relevant_set = set(relevant_items)

    return len(predicted_top_k.intersection(relevant_set)) / k


def recall_at_k(predicted_items, relevant_items, k):
    predicted_top_k = set(predicted_items[:k])
    relevant_set = set(relevant_items)

    return len(
        predicted_top_k.intersection(relevant_set)
    ) / len(relevant_set)


def dcg_at_k(relevance_scores, k):
    relevance_scores = np.asarray(relevance_scores)[:k]

    if len(relevance_scores) == 0:
        return 0.0

    discounts = np.log2(
        np.arange(2, len(relevance_scores) + 2)
    )

    return np.sum(
        (2 ** relevance_scores - 1) / discounts
    )


def ndcg_at_k(relevance_scores, k):
    relevance_scores = np.asarray(relevance_scores)

    actual_dcg = dcg_at_k(relevance_scores, k)

    ideal_scores = np.sort(
        relevance_scores
    )[::-1]

    ideal_dcg = dcg_at_k(
        ideal_scores,
        k
    )

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

In [57]:
ranking_predictions['item_id'] = (
    ranking_predictions['client_hash_id']
    + '__'
    + ranking_predictions['content_hash_id']
)

In [58]:
predicted_ranking = ranking_predictions.sort_values(
    'predicted_opportunity_score',
    ascending=False
).reset_index(drop=True)

future_ranking = ranking_predictions.sort_values(
    'future_opportunity_score',
    ascending=False
).reset_index(drop=True)

In [59]:
top_k_values = [10, 25, 50, 100]

future_top_k = {}

for k in top_k_values:
    future_top_k[k] = set(
        future_ranking.head(k)['item_id']
    )

In [60]:
model_metrics = []

for k in top_k_values:

    predicted_items = predicted_ranking['item_id'].tolist()
    relevant_items = future_top_k[k]

    precision = precision_at_k(
        predicted_items,
        relevant_items,
        k
    )

    recall = recall_at_k(
        predicted_items,
        relevant_items,
        k
    )

    top_k_predicted = predicted_items[:k]

    relevance_scores = [
        1 if item in relevant_items else 0
        for item in top_k_predicted
    ]

    ndcg = ndcg_at_k(
        relevance_scores,
        k
    )

    model_metrics.append({
        'k': k,
        'precision_at_k': precision,
        'recall_at_k': recall,
        'ndcg_at_k': ndcg
    })

model_metrics_df = pd.DataFrame(model_metrics)

model_metrics_df

,k,precision_at_k,recall_at_k,ndcg_at_k
0,10,0.60,0.60,1.000000
1,25,0.52,0.52,0.972151
2,50,0.72,0.72,0.969789
3,100,0.61,0.61,0.975610


In [61]:
baseline_historical_impressions = (
    modeling_dataset[
        [
            'client_hash_id',
            'content_hash_id',
            'historical_impressions'
        ]
    ]
    .copy()
)

baseline_historical_impressions['item_id'] = (
    baseline_historical_impressions['client_hash_id']
    + '__'
    + baseline_historical_impressions['content_hash_id']
)

baseline_historical_impressions = (
    baseline_historical_impressions
    .sort_values(
        'historical_impressions',
        ascending=False
    )
    .reset_index(drop=True)
)

In [65]:
baseline_historical_clicks = (
    modeling_dataset[
        [
            'client_hash_id',
            'content_hash_id',
            'historical_clicks'
        ]
    ]
    .copy()
)

baseline_historical_clicks['item_id'] = (
    baseline_historical_clicks['client_hash_id']
    + '__'
    + baseline_historical_clicks['content_hash_id']
)

baseline_historical_clicks = (
    baseline_historical_clicks
    .sort_values(
        'historical_clicks',
        ascending=False
    )
    .reset_index(drop=True)
)

In [66]:
baseline_historical_position = (
    modeling_dataset[
        [
            'client_hash_id',
            'content_hash_id',
            'historical_mean_position'
        ]
    ]
    .copy()
)

baseline_historical_position = (
    baseline_historical_position[
        baseline_historical_position[
            'historical_mean_position'
        ].notna()
        &
        (
            baseline_historical_position[
                'historical_mean_position'
            ] > 0
        )
    ]
)

baseline_historical_position['item_id'] = (
    baseline_historical_position['client_hash_id']
    + '__'
    + baseline_historical_position['content_hash_id']
)

baseline_historical_position = (
    baseline_historical_position
    .sort_values(
        'historical_mean_position',
        ascending=True
    )
    .reset_index(drop=True)
)

In [64]:
baseline_historical_position = (
    modeling_dataset[
        [
            'client_hash_id',
            'content_hash_id',
            'historical_mean_position'
        ]
    ]
    .copy()
)

baseline_historical_position = (
    baseline_historical_position[
        baseline_historical_position[
            'historical_mean_position'
        ].notna()
        &
        (
            baseline_historical_position[
                'historical_mean_position'
            ] > 0
        )
    ]
)

baseline_historical_position['item_id'] = (
    baseline_historical_position['client_hash_id']
    + '__'
    + baseline_historical_position['content_hash_id']
)

baseline_historical_position = (
    baseline_historical_position
    .sort_values(
        'historical_mean_position',
        ascending=True
    )
    .reset_index(drop=True)
)

In [67]:
def evaluate_ranking(
    ranking_df,
    score_column,
    future_top_k,
    top_k_values
):

    results = []

    ranked_items = ranking_df['item_id'].tolist()

    for k in top_k_values:

        relevant_items = future_top_k[k]

        predicted_top_k = ranked_items[:k]

        hits = len(
            set(predicted_top_k)
            .intersection(relevant_items)
        )

        precision = hits / k
        recall = hits / k

        relevance_scores = [
            1 if item in relevant_items else 0
            for item in predicted_top_k
        ]

        ndcg = ndcg_at_k(
            relevance_scores,
            k
        )

        results.append({
            'k': k,
            'precision_at_k': precision,
            'recall_at_k': recall,
            'ndcg_at_k': ndcg
        })

    return pd.DataFrame(results)

In [68]:
impression_metrics = evaluate_ranking(
    baseline_historical_impressions,
    'historical_impressions',
    future_top_k,
    top_k_values
)

impression_metrics

,k,precision_at_k,recall_at_k,ndcg_at_k
0,10,0.00,0.00,0.000000
1,25,0.04,0.04,0.239812
2,50,0.04,0.04,0.266579
3,100,0.06,0.06,0.326966


In [69]:
click_metrics = evaluate_ranking(
    baseline_historical_clicks,
    'historical_clicks',
    future_top_k,
    top_k_values
)

click_metrics

,k,precision_at_k,recall_at_k,ndcg_at_k
0,10,0.0,0.0,0.0
1,25,0.0,0.0,0.0
2,50,0.0,0.0,0.0
3,100,0.0,0.0,0.0


In [70]:
position_metrics = evaluate_ranking(
    baseline_historical_position,
    'historical_mean_position',
    future_top_k,
    top_k_values
)

position_metrics

,k,precision_at_k,recall_at_k,ndcg_at_k
0,10,0.0,0.0,0.0
1,25,0.0,0.0,0.0
2,50,0.0,0.0,0.0
3,100,0.0,0.0,0.0


In [71]:
model_eval = model_metrics_df.copy()
model_eval['method'] = 'Random Forest'

impression_eval = impression_metrics.copy()
impression_eval['method'] = 'Historical Impressions'

click_eval = click_metrics.copy()
click_eval['method'] = 'Historical Clicks'

position_eval = position_metrics.copy()
position_eval['method'] = 'Historical Position'

evaluation_results = pd.concat(
    [
        model_eval,
        impression_eval,
        click_eval,
        position_eval
    ],
    ignore_index=True
)

evaluation_results = evaluation_results[
    [
        'method',
        'k',
        'precision_at_k',
        'recall_at_k',
        'ndcg_at_k'
    ]
]

evaluation_results

,method,k,precision_at_k,recall_at_k,ndcg_at_k
0,Random Forest,10,0.60,0.60,1.000000
1,Random Forest,25,0.52,0.52,0.972151
2,Random Forest,50,0.72,0.72,0.969789
3,Random Forest,100,0.61,0.61,0.975610
4,Historical Impressions,10,0.00,0.00,0.000000
5,Historical Impressions,25,0.04,0.04,0.239812
6,Historical Impressions,50,0.04,0.04,0.266579
7,Historical Impressions,100,0.06,0.06,0.326966
8,Historical Clicks,10,0.00,0.00,0.000000
9,Historical Clicks,25,0.00,0.00,0.000000


In [72]:
model_items = set(
    ranking_predictions['item_id']
)

impression_items = set(
    baseline_historical_impressions['item_id']
)

click_items = set(
    baseline_historical_clicks['item_id']
)

position_items = set(
    baseline_historical_position['item_id']
)

future_items = set(
    future_ranking['item_id']
)

print("Model items:", len(model_items))
print("Historical impressions items:", len(impression_items))
print("Historical clicks items:", len(click_items))
print("Historical position items:", len(position_items))
print("Future benchmark items:", len(future_items))

Model items: 64765
Historical impressions items: 64765
Historical clicks items: 64765
Historical position items: 63775
Future benchmark items: 64765


In [73]:
future_top100 = set(
    future_ranking.head(100)['item_id']
)

print(
    "Historical impressions overlap:",
    len(
        impression_items.intersection(future_top100)
    )
)

print(
    "Historical clicks overlap:",
    len(
        click_items.intersection(future_top100)
    )
)

print(
    "Historical position overlap:",
    len(
        position_items.intersection(future_top100)
    )
)

Historical impressions overlap: 100
Historical clicks overlap: 100
Historical position overlap: 97


In [74]:
print("Historical Impressions:")
display(
    baseline_historical_impressions[
        [
            'client_hash_id',
            'content_hash_id',
            'historical_impressions'
        ]
    ].head(10)
)

print("\nHistorical Clicks:")
display(
    baseline_historical_clicks[
        [
            'client_hash_id',
            'content_hash_id',
            'historical_clicks'
        ]
    ].head(10)
)

print("\nHistorical Position:")
display(
    baseline_historical_position[
        [
            'client_hash_id',
            'content_hash_id',
            'historical_mean_position'
        ]
    ].head(10)
)

Historical Impressions:


,client_hash_id,content_hash_id,historical_impressions
0,client_e547b89c05043229,content_963de14b1f58978f,505588.0
1,client_e547b89c05043229,content_eadb33b5df496f4a,378809.0
2,client_e547b89c05043229,content_545bb6cc7081ded3,343807.0
3,client_8ddc46da5414ffd8,content_943dc881428182b8,271667.0
4,client_9c26c096d6e57253,content_adcc7b85a04c187d,216039.0
5,client_8ddc46da5414ffd8,content_b902320872acab45,211686.0
6,client_e547b89c05043229,content_9ef3d7516483e665,200514.0
7,client_9c26c096d6e57253,content_54f2b96801c90591,199602.0
8,client_e547b89c05043229,content_cc26620b2cbb837f,192459.0
9,client_8ddc46da5414ffd8,content_32c5cc913fb4ff41,191990.0



Historical Clicks:


,client_hash_id,content_hash_id,historical_clicks
0,client_9c26c096d6e57253,content_adcc7b85a04c187d,115329.0
1,client_9c26c096d6e57253,content_54f2b96801c90591,105771.0
2,client_e547b89c05043229,content_eadb33b5df496f4a,2967.0
3,client_e547b89c05043229,content_545bb6cc7081ded3,2032.0
4,client_e547b89c05043229,content_963de14b1f58978f,1471.0
5,client_b77d0d5f08f05e64,content_ac1ddc0c0e79289f,1169.0
6,client_8ddc46da5414ffd8,content_d46321b2dce9da21,1027.0
7,client_0fa64a184f18a4a0,content_2db2a9dcb3b62a3a,938.0
8,client_0fa64a184f18a4a0,content_2bed2c9ce7808050,919.0
9,client_e5c2aa26a8598242,content_36fab41f31ba7825,893.0



Historical Position:


,client_hash_id,content_hash_id,historical_mean_position
0,client_9958f0a7ae1df715,content_d802db79952aed99,0.666667
1,client_8ddc46da5414ffd8,content_d584c27d7f8ded3b,0.838047
2,client_8ddc46da5414ffd8,content_39617cae39df669a,0.913907
3,client_62f4a7e64f5e0096,content_8022d335a0934e80,0.915135
4,client_73cda7b4e4f265ea,content_de9e46db895234a3,0.955215
5,client_62f4a7e64f5e0096,content_674f272dbd3142d9,0.986147
6,client_e547b89c05043229,content_f43b95944a9fa212,0.986461
7,client_a22068e339bf95f5,content_fe7af769ad568681,1.000000
8,client_a22068e339bf95f5,content_4226eaa46864a67f,1.000000
9,client_b10cb2997d0c7c86,content_ea78e8d9e0e5f03e,1.000000


In [76]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training samples: 51812
Test samples: 12953
Training target: (51812,)
Test target: (12953,)


In [77]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("X_train_prepared shape:", X_train_prepared.shape)
print("X_test_prepared shape:", X_test_prepared.shape)

X_train_prepared shape: (51812, 26)
X_test_prepared shape: (12953, 26)


In [78]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_prepared, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [79]:
test_predictions = rf_model.predict(X_test_prepared)

print("Prediction count:", len(test_predictions))
print("Prediction mean:", test_predictions.mean())
print("Prediction std:", test_predictions.std())
print("Prediction min:", test_predictions.min())
print("Prediction max:", test_predictions.max())

Prediction count: 12953
Prediction mean: 0.5467972218588399
Prediction std: 0.0644795852704824
Prediction min: 0.4427190253990353
Prediction max: 0.8168941198228551


In [80]:
test_indices = X_test.index

test_ranking = modeling_dataset.loc[
    test_indices,
    ['client_hash_id',
     'content_hash_id',
     'future_opportunity_score',
     'future_reference_rank']
].copy()

test_ranking['predicted_opportunity_score'] = test_predictions

test_ranking['item_id'] = (
    test_ranking['client_hash_id']
    + '__'
    + test_ranking['content_hash_id']
)

test_ranking['model_rank'] = (
    test_ranking['predicted_opportunity_score']
    .rank(method='first', ascending=False)
    .astype(int)
)

test_ranking = test_ranking.sort_values(
    'predicted_opportunity_score',
    ascending=False
)

test_ranking.head(10)

,client_hash_id,content_hash_id,future_opportunity_score,future_reference_rank,predicted_opportunity_score,item_id,model_rank
41818,client_23a62021009f63c4,content_6aa54d6bbdbf6f24,0.840947,6,0.816894,client_23a62021009f63c4__content_6aa54d6bbdbf6f24,1
35839,client_3197e6291363b4db,content_3fd3671dd5e2604f,0.844297,4,0.808748,client_3197e6291363b4db__content_3fd3671dd5e2604f,2
14769,client_23a62021009f63c4,content_2de9a39d3482a269,0.813685,15,0.796337,client_23a62021009f63c4__content_2de9a39d3482a269,3
52231,client_73cda7b4e4f265ea,content_3e56218fa52d24b8,0.817884,12,0.790494,client_73cda7b4e4f265ea__content_3e56218fa52d24b8,4
23096,client_3197e6291363b4db,content_9ab64254b0acf205,0.798155,31,0.790282,client_3197e6291363b4db__content_9ab64254b0acf205,5
754,client_e547b89c05043229,content_30d414c2defff507,0.797235,32,0.784161,client_e547b89c05043229__content_30d414c2defff507,6
62190,client_e547b89c05043229,content_fe68417a6bf95bce,0.744194,369,0.783370,client_e547b89c05043229__content_fe68417a6bf95bce,7
37639,client_23a62021009f63c4,content_2409eee8ef6f56dc,0.805735,24,0.781376,client_23a62021009f63c4__content_2409eee8ef6f56dc,8
39935,client_23a62021009f63c4,content_c20a5d57d6cd25f1,0.791369,42,0.780938,client_23a62021009f63c4__content_c20a5d57d6cd25f1,9
50208,client_23a62021009f63c4,content_4704e13e0c70d8e8,0.796429,33,0.779683,client_23a62021009f63c4__content_4704e13e0c70d8e8,10


In [81]:
from scipy.stats import spearmanr

spearman_corr, spearman_p = spearmanr(
    test_ranking['predicted_opportunity_score'],
    test_ranking['future_opportunity_score']
)

print("Test Spearman correlation:", spearman_corr)
print("P-value:", spearman_p)

Test Spearman correlation: 0.8277043403189904
P-value: 0.0


In [82]:
import numpy as np

def precision_at_k(predicted_items, relevant_items, k):
    predicted_top_k = predicted_items[:k]
    return len(set(predicted_top_k).intersection(relevant_items)) / k


def recall_at_k(predicted_items, relevant_items, k):
    predicted_top_k = predicted_items[:k]
    return len(set(predicted_top_k).intersection(relevant_items)) / len(relevant_items)


def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    
    if len(relevances) == 0:
        return 0.0
    
    discounts = np.log2(np.arange(2, len(relevances) + 2))
    
    return np.sum(
        (2 ** relevances - 1) / discounts
    )


def ndcg_at_k(predicted_items, future_scores, k):
    score_lookup = dict(
        zip(
            future_scores['item_id'],
            future_scores['future_opportunity_score']
        )
    )
    
    predicted_scores = [
        score_lookup.get(item_id, 0)
        for item_id in predicted_items[:k]
    ]
    
    ideal_scores = sorted(
        score_lookup.values(),
        reverse=True
    )[:k]
    
    predicted_dcg = dcg_at_k(predicted_scores, k)
    ideal_dcg = dcg_at_k(ideal_scores, k)
    
    if ideal_dcg == 0:
        return 0.0
    
    return predicted_dcg / ideal_dcg

In [83]:
future_ranking_test = (
    test_ranking
    .sort_values('future_opportunity_score', ascending=False)
    .reset_index(drop=True)
)

predicted_ranking_test = (
    test_ranking
    .sort_values('predicted_opportunity_score', ascending=False)
    .reset_index(drop=True)
)

future_items_test = set(
    future_ranking_test['item_id']
)

predicted_items_test = list(
    predicted_ranking_test['item_id']
)

print("Test items:", len(test_ranking))
print("Future benchmark items:", len(future_items_test))

Test items: 12953
Future benchmark items: 12953


In [84]:
evaluation_rows = []

for k in [10, 25, 50, 100]:
    
    future_top_k = set(
        future_ranking_test
        .head(k)['item_id']
    )
    
    predicted_top_k = set(
        predicted_ranking_test
        .head(k)['item_id']
    )
    
    overlap = len(
        predicted_top_k.intersection(future_top_k)
    )
    
    precision = overlap / k
    recall = overlap / k
    
    ndcg = ndcg_at_k(
        predicted_items_test,
        future_ranking_test,
        k
    )
    
    evaluation_rows.append({
        'method': 'Random Forest',
        'k': k,
        'precision': precision,
        'recall': recall,
        'ndcg': ndcg
    })

evaluation_results_test = pd.DataFrame(
    evaluation_rows
)

evaluation_results_test

,method,k,precision,recall,ndcg
0,Random Forest,10,0.50,0.50,0.983057
1,Random Forest,25,0.64,0.64,0.970115
2,Random Forest,50,0.56,0.56,0.969419
3,Random Forest,100,0.55,0.55,0.969281


In [85]:
# -----------------------------------------
# Prepare Historical Baselines for Test Set
# -----------------------------------------

test_item_ids = set(test_ranking['item_id'])

baseline_impressions_test = (
    baseline_historical_impressions[
        baseline_historical_impressions['item_id'].isin(test_item_ids)
    ]
    .sort_values('historical_impressions', ascending=False)
    .reset_index(drop=True)
)

baseline_clicks_test = (
    baseline_historical_clicks[
        baseline_historical_clicks['item_id'].isin(test_item_ids)
    ]
    .sort_values('historical_clicks', ascending=False)
    .reset_index(drop=True)
)

baseline_position_test = (
    baseline_historical_position[
        baseline_historical_position['item_id'].isin(test_item_ids)
    ]
    .sort_values('historical_mean_position', ascending=True)
    .reset_index(drop=True)
)

print("Historical impressions:", len(baseline_impressions_test))
print("Historical clicks:", len(baseline_clicks_test))
print("Historical position:", len(baseline_position_test))

Historical impressions: 12953
Historical clicks: 12953
Historical position: 12764


In [86]:
# --------------------
# Evaluation Function
# --------------------

def evaluate_baseline(
    baseline_df,
    score_column,
    method_name,
    ascending=False
):
    
    ranked_df = (
        baseline_df
        .sort_values(score_column, ascending=ascending)
        .reset_index(drop=True)
    )
    
    predicted_items = list(ranked_df['item_id'])
    
    results = []
    
    for k in [10, 25, 50, 100]:
        
        future_top_k = set(
            future_ranking_test
            .head(k)['item_id']
        )
        
        predicted_top_k = set(
            predicted_items[:k]
        )
        
        overlap = len(
            predicted_top_k.intersection(future_top_k)
        )
        
        precision = overlap / k
        recall = overlap / k
        
        ndcg = ndcg_at_k(
            predicted_items,
            future_ranking_test,
            k
        )
        
        results.append({
            'method': method_name,
            'k': k,
            'precision': precision,
            'recall': recall,
            'ndcg': ndcg
        })
    
    return pd.DataFrame(results)

In [87]:
# ----------------------------
# Evaluate the Three Baselines
# ----------------------------

impressions_results = evaluate_baseline(
    baseline_impressions_test,
    'historical_impressions',
    'Historical Impressions',
    ascending=False
)

clicks_results = evaluate_baseline(
    baseline_clicks_test,
    'historical_clicks',
    'Historical Clicks',
    ascending=False
)

position_results = evaluate_baseline(
    baseline_position_test,
    'historical_mean_position',
    'Historical Position',
    ascending=True
)

In [88]:
# -------------------------
# ---Combine All Results---
# -------------------------

model_comparison = pd.concat(
    [
        evaluation_results_test,
        impressions_results,
        clicks_results,
        position_results
    ],
    ignore_index=True
)

model_comparison

,method,k,precision,recall,ndcg
0,Random Forest,10,0.50,0.50,0.983057
1,Random Forest,25,0.64,0.64,0.970115
2,Random Forest,50,0.56,0.56,0.969419
3,Random Forest,100,0.55,0.55,0.969281
4,Historical Impressions,10,0.00,0.00,0.749582
5,Historical Impressions,25,0.04,0.04,0.770668
6,Historical Impressions,50,0.06,0.06,0.792397
7,Historical Impressions,100,0.09,0.09,0.807323
8,Historical Clicks,10,0.00,0.00,0.748636
9,Historical Clicks,25,0.00,0.00,0.739255


In [89]:
model_comparison.sort_values(
    ['k', 'method']
).reset_index(drop=True)

,method,k,precision,recall,ndcg
0,Historical Clicks,10,0.00,0.00,0.748636
1,Historical Impressions,10,0.00,0.00,0.749582
2,Historical Position,10,0.00,0.00,0.516845
3,Random Forest,10,0.50,0.50,0.983057
4,Historical Clicks,25,0.00,0.00,0.739255
5,Historical Impressions,25,0.04,0.04,0.770668
6,Historical Position,25,0.00,0.00,0.529164
7,Random Forest,25,0.64,0.64,0.970115
8,Historical Clicks,50,0.00,0.00,0.748774
9,Historical Impressions,50,0.06,0.06,0.792397


In [90]:
comparison_summary = (
    model_comparison
    .sort_values(['k', 'ndcg'], ascending=[True, False])
    .groupby('k', as_index=False)
    .first()
)

comparison_summary

,k,method,precision,recall,ndcg
0,10,Random Forest,0.50,0.50,0.983057
1,25,Random Forest,0.64,0.64,0.970115
2,50,Random Forest,0.56,0.56,0.969419
3,100,Random Forest,0.55,0.55,0.969281


In [91]:
comparison_summary = (
    model_comparison
    .sort_values(['k', 'ndcg'], ascending=[True, False])
    .groupby('k', as_index=False)
    .first()
)

comparison_summary

,k,method,precision,recall,ndcg
0,10,Random Forest,0.50,0.50,0.983057
1,25,Random Forest,0.64,0.64,0.970115
2,50,Random Forest,0.56,0.56,0.969419
3,100,Random Forest,0.55,0.55,0.969281


In [92]:
rf_summary = evaluation_results_test.copy()

rf_summary

,method,k,precision,recall,ndcg
0,Random Forest,10,0.50,0.50,0.983057
1,Random Forest,25,0.64,0.64,0.970115
2,Random Forest,50,0.56,0.56,0.969419
3,Random Forest,100,0.55,0.55,0.969281


## 10.8 Model Comparison

The Random Forest model was evaluated against simple historical-performance baselines using an out-of-sample test set.

The model achieved a Spearman correlation of 0.828 between predicted opportunity scores and the future reference opportunity scores, indicating strong agreement in the relative ordering of content items.

At the top-10 level, the model achieved 0.50 Precision@10 and 0.983 NDCG@10. At larger ranking cutoffs, Precision@25 reached 0.64, while Precision@50 and Precision@100 were 0.56 and 0.55 respectively.

The historical-performance baselines provide simple reference points for comparison. Historical impressions, clicks, and position represent single-signal ranking strategies, whereas the Random Forest combines historical performance with content-level characteristics.

These results suggest that combining multiple historical performance signals with content characteristics can produce a ranking that aligns substantially with the future opportunity benchmark.

However, the opportunity score used for evaluation is a reference benchmark derived from observable future performance rather than a ground-truth label of successful content improvement. Therefore, the model should be interpreted as a prioritization and ranking approach rather than a predictor of guaranteed improvement outcomes.

## 10.9 Phase 10 Conclusion

Phase 10 developed and evaluated a Random Forest ranking model for prioritizing content items according to their relative opportunity for review or improvement.

The modeling process used historical information from June 1–20, 2026 as input features and evaluated the predicted rankings against a future opportunity benchmark calculated from June 21–30, 2026. This temporal setup ensured that future performance information was not used as a model feature.

The final Random Forest model used historical search-performance metrics together with content-level characteristics such as search volume, competition, backlinks, content length, content type, and search intent.

The model was evaluated using an out-of-sample test set containing 12,953 content items. It achieved a Spearman correlation of 0.828 between predicted opportunity scores and future opportunity scores.

Ranking evaluation showed the following results:

- Precision@10 = 0.50
- Precision@25 = 0.64
- Precision@50 = 0.56
- Precision@100 = 0.55
- NDCG@10 = 0.983
- NDCG@25 = 0.970
- NDCG@50 = 0.969
- NDCG@100 = 0.969

Compared with simple historical-performance baselines, the Random Forest produced substantially higher ranking alignment with the future opportunity benchmark. Historical impressions achieved Precision@100 of 0.09, while historical clicks and historical position achieved 0.00 Precision@100.

These results indicate that combining multiple historical performance signals with content-level characteristics can provide a useful data-driven prioritization approach compared with relying on a single historical metric.

However, the future opportunity score is a reference benchmark derived from observable performance signals rather than a ground-truth label indicating whether a content optimization was successful. Therefore, the model should be interpreted as a content prioritization and ranking system, not as a causal model or a predictor of guaranteed improvements in rankings, traffic, or clicks.

The next phase should focus on interpreting the model's drivers, identifying the characteristics associated with high predicted opportunity, and converting the ranking output into actionable content-priority recommendations.